In [0]:
%run ../source_to_bronze/utils

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
employee_bronze_path = "/Volumes/assignment/bronze/raw_data/source_to_bronze/employee"

department_bronze_path = "/Volumes/assignment/bronze/raw_data/source_to_bronze/department"

country_bronze_path = "/Volumes/assignment/bronze/raw_data/source_to_bronze/country"

In [0]:
employee_bronze_df.printSchema()

In [0]:
employee_schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("EmployeeName", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Age", IntegerType(), True)
])

In [0]:
employee_bronze_df.printSchema()

In [0]:
department_schema = StructType([
    StructField("DepartmentID", StringType(), True),
    StructField("DepartmentName", StringType(), True)
])

In [0]:
country_schema = StructType([
    StructField("CountryCode", StringType(), True),
    StructField("CountryName", StringType(), True)
])

In [0]:
employee_df = (
    spark.read
    .option("header", True)
    .schema(employee_schema)
    .csv(employee_bronze_path)
)

display(employee_df)

In [0]:
department_df = (
    spark.read
    .option("header", True)
    .schema(department_schema)
    .csv(department_bronze_path)
)

display(department_df)

In [0]:
country_df = (
    spark.read
    .option("header", True)
    .schema(country_schema)
    .csv(country_bronze_path)
)

display(country_df)

In [0]:
employee_df = employee_df.toDF(
    *[camel_to_snake(col) for col in employee_df.columns]
)

In [0]:
employee_df.printSchema()

In [0]:
department_df = department_df.toDF(
    *[camel_to_snake(col) for col in department_df.columns]
)

In [0]:
country_df = country_df.toDF(
    *[camel_to_snake(col) for col in country_df.columns]
)

In [0]:
employee_df = employee_df.withColumn(
    "load_date",
    F.current_date()
)

In [0]:
department_df = department_df.withColumn(
    "load_date",
    F.current_date()
)

In [0]:
country_df = country_df.withColumn(
    "load_date",
    F.current_date()
)

In [0]:
display(employee_df)

In [0]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS Employee_info
""")

In [0]:
silver_employee_path = "/Volumes/assignment/bronze/raw_data/bronze_to_silver/dim_employee"

In [0]:
(
    employee_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Employee_info.dim_employee")
)

In [0]:
silver_employee_df = spark.table("Employee_info.dim_employee")

display(silver_employee_df)